# 🌐 Random Surfer Model — Without Teleportation

This notebook explores the **Random Surfer Model** (the foundation of PageRank) *without* the teleportation (damping) fix. We deliberately expose the pathological behaviours that arise in real web graphs:

| Problem | Cause | Symptom |
|---|---|---|
| **Probability leak** | Dangling nodes (no out-edges) | Rank mass drains to zero |
| **Rank traps** | Cycles / sink SCCs | Rank pools incorrectly in loops |
| **Non-convergence** | Reducible graph structure | Power iteration never settles |

We build a 100-node graph that deliberately contains all three issues, then surgically observe each pathology.

## 0 · Imports & Style

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import warnings, time
warnings.filterwarnings('ignore')

# ── Consistent aesthetics ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#161b22',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#c9d1d9',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'text.color'       : '#c9d1d9',
    'grid.color'       : '#21262d',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'font.family'      : 'monospace',
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
})

ACCENT   = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']
rng = np.random.default_rng(42)
print('Libraries loaded ✓')

## 1 · Build the Web Graph (100 nodes)

We construct a graph with three deliberately-crafted regions:

- **Region A** (nodes 0–49): A sparse random directed graph — the "normal" web.
- **Region B** (nodes 50–69): A tight cycle (rank trap / sink SCC).
- **Region C** (nodes 70–79): Dangling nodes — zero out-edges, pure probability sinks.
- **Connective tissue**: A few cross-region edges so the graph is weakly connected.

In [ ]:
N = 100
G = nx.DiGraph()
G.add_nodes_from(range(N))

# ── Region A: sparse random graph (nodes 0–49) ────────────────────────────────
for u in range(50):
    n_out = rng.integers(1, 6)          # 1–5 out-edges each
    targets = rng.choice([v for v in range(50) if v != u], size=n_out, replace=False)
    for v in targets:
        G.add_edge(u, v)

# ── Region B: tight cycle of length 20 (nodes 50–69) ─────────────────────────
cycle_nodes = list(range(50, 70))
for i, u in enumerate(cycle_nodes):
    v = cycle_nodes[(i + 1) % len(cycle_nodes)]
    G.add_edge(u, v)
# A few intra-cycle chords to make it more realistic
G.add_edge(50, 60); G.add_edge(55, 65); G.add_edge(63, 53)

# ── Region C: dangling nodes (nodes 70–79) — NO out-edges ────────────────────
dangling_nodes = list(range(70, 80))
# Only in-edges from region A
for d in dangling_nodes:
    src = rng.choice(range(50))
    G.add_edge(src, d)

# ── Connective tissue: A→B, A→C, B→A edges ───────────────────────────────────
for _ in range(8):
    G.add_edge(rng.integers(0, 50), rng.integers(50, 70))  # A → B
for _ in range(5):
    G.add_edge(rng.integers(50, 70), rng.integers(0, 50))  # B → A

# ── Remaining nodes 80–99: a small isolated-ish cluster ──────────────────────
for u in range(80, 100):
    targets = rng.choice([v for v in range(80, 100) if v != u], size=2, replace=False)
    for v in targets:
        G.add_edge(u, v)
# Bridge from region A into cluster
G.add_edge(5, 85); G.add_edge(20, 90)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Dangling nodes (out-degree 0): {[n for n in G.nodes if G.out_degree(n) == 0]}')
print(f'Is strongly connected: {nx.is_strongly_connected(G)}')
sccs = list(nx.strongly_connected_components(G))
print(f'Number of SCCs: {len(sccs)}')
print(f'Largest SCC size: {max(len(s) for s in sccs)}')

### 1.1 · Visualise the Graph Structure

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Colour nodes by region
color_map = []
for n in G.nodes:
    if n < 50:      color_map.append(ACCENT[0])   # blue  — normal
    elif n < 70:    color_map.append(ACCENT[2])   # red   — cycle trap
    elif n < 80:    color_map.append(ACCENT[4])   # orange — dangling
    else:           color_map.append(ACCENT[3])   # purple — isolated cluster

pos = nx.spring_layout(G, seed=7, k=0.35)
nx.draw_networkx_nodes(G, pos, node_color=color_map, node_size=120, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#30363d', arrows=True,
                       arrowsize=8, width=0.6, alpha=0.7, ax=ax,
                       connectionstyle='arc3,rad=0.05')

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(facecolor=ACCENT[0], label='Region A — normal (0–49)'),
    Patch(facecolor=ACCENT[2], label='Region B — cycle trap (50–69)'),
    Patch(facecolor=ACCENT[4], label='Region C — dangling (70–79)'),
    Patch(facecolor=ACCENT[3], label='Cluster — semi-isolated (80–99)'),
]
ax.legend(handles=legend, loc='upper left', framealpha=0.3, fontsize=9)
ax.set_title('Web Graph — 100 nodes  |  Reducible & Cyclic', pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2 · Build the Transition Matrix (No Teleportation)

The standard row-stochastic transition matrix **H** is built by normalising each row by its out-degree.  
**Dangling rows are left as all-zeros** — this is exactly the bug we want to expose.

In [ ]:
def build_transition_matrix(G):
    """Return H (N×N) row-stochastic, dangling rows = 0."""
    N = G.number_of_nodes()
    H = np.zeros((N, N))
    for u in G.nodes:
        out_edges = list(G.successors(u))
        if out_edges:
            for v in out_edges:
                H[u, v] = 1.0 / len(out_edges)
    return H

H = build_transition_matrix(G)

dangling = [n for n in G.nodes if G.out_degree(n) == 0]
print(f'Transition matrix H shape: {H.shape}')
print(f'Dangling nodes (zero rows): {dangling}')
print(f'Row sums — min: {H.sum(axis=1).min():.4f},  max: {H.sum(axis=1).max():.4f}')
print(f'Total probability mass in H: {H.sum():.4f}  (should equal N - |dangling| = {N - len(dangling)})')

## 3 · Power Iteration — Full Run

We run power iteration starting from the **uniform distribution** and track the *total probability mass* and per-node rank vector at every step.

In [ ]:
def power_iteration(H, max_iter=300, tol=1e-10, track_nodes=None):
    """
    Run power iteration on transition matrix H.
    Returns history of rank vectors (and tracked node histories).
    """
    N = H.shape[0]
    r = np.ones(N) / N          # uniform start
    history = [r.copy()]
    total_mass = [r.sum()]
    node_history = {n: [r[n]] for n in (track_nodes or [])}

    for _ in range(max_iter):
        r_new = r @ H           # r^T * H  (row vector × matrix)
        history.append(r_new.copy())
        total_mass.append(r_new.sum())
        for n in (track_nodes or []):
            node_history[n].append(r_new[n])
        if np.linalg.norm(r_new - r, 1) < tol:
            break
        r = r_new

    return np.array(history), np.array(total_mass), node_history

# Nodes to watch:
#   5  — normal node in region A
#   55 — inside the cycle trap
#   73 — a dangling node
#   85 — semi-isolated cluster
watch = [5, 55, 73, 85]

history, total_mass, node_hist = power_iteration(H, max_iter=300, track_nodes=watch)
print(f'Iterations run: {len(history) - 1}')
print(f'Final total probability mass: {total_mass[-1]:.6f}  (started at 1.0)')
print(f'Probability lost: {1 - total_mass[-1]:.6f}')

## 4 · Pathology 1 — Probability Leak from Dangling Nodes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

iters = np.arange(len(total_mass))

# ── Left: total mass decay ────────────────────────────────────────────────────
ax = axes[0]
ax.plot(iters, total_mass, color=ACCENT[2], lw=2.5, label='Total probability mass')
ax.axhline(1.0, color='#8b949e', ls='--', lw=1, label='Initial mass = 1')
ax.fill_between(iters, total_mass, 1.0, alpha=0.15, color=ACCENT[2], label='Leaked probability')
ax.set_xlabel('Iteration')
ax.set_ylabel('∑ rank(v)')
ax.set_title('Probability Leak — Total Mass Over Iterations')
ax.legend(fontsize=9)
ax.grid(True)

# ── Right: dangling nodes absorbing and then losing mass ─────────────────────
ax = axes[1]
for d in dangling_nodes[:5]:   # show first 5 dangling nodes
    dh = [history[t][d] for t in range(len(history))]
    ax.plot(iters, dh, lw=1.5, alpha=0.85)

# Highlight the peak
ex_d = dangling_nodes[0]
dh0 = [history[t][ex_d] for t in range(len(history))]
peak_iter = int(np.argmax(dh0))
ax.annotate(f'Rank peaks then drains\nto 0 (node {ex_d})',
            xy=(peak_iter, dh0[peak_iter]),
            xytext=(peak_iter + 20, dh0[peak_iter] * 1.5),
            arrowprops=dict(arrowstyle='->', color=ACCENT[4]),
            color=ACCENT[4], fontsize=9)
ax.set_xlabel('Iteration')
ax.set_ylabel('rank(v)')
ax.set_title('Dangling Nodes (70–79) — Rank Over Time')
ax.grid(True)

plt.suptitle('Pathology 1: Probability Leaks Out Through Dangling Nodes',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nDangling nodes contribute to leak:')
print(f'  Each dangling node absorbs incoming rank but distributes nothing.')
print(f'  Over iterations, that mass simply vanishes from the system.')

## 5 · Pathology 2 — Rank Traps & Cycle Pumping

The cycle (nodes 50–69) forms a **sink SCC**: probability flows in but very little flows out. Over iterations the cycle accumulates disproportionate rank — a fundamentally wrong result.

In [ ]:
# Track all cycle nodes and a few normal nodes
cycle_nodes_watch = list(range(50, 70))
normal_nodes_watch = [0, 5, 10, 20, 30]

# Compute mean rank within each region over iterations
region_A_mass = [history[t][:50].sum()  for t in range(len(history))]
region_B_mass = [history[t][50:70].sum() for t in range(len(history))]
region_C_mass = [history[t][70:80].sum() for t in range(len(history))]
cluster_mass  = [history[t][80:].sum()   for t in range(len(history))]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

iters = np.arange(len(history))

# ── Top-left: Region mass over time ──────────────────────────────────────────
ax = axes[0, 0]
ax.plot(iters, region_A_mass, color=ACCENT[0], lw=2, label='Region A (normal, 50 nodes)')
ax.plot(iters, region_B_mass, color=ACCENT[2], lw=2, label='Region B (cycle trap, 20 nodes)')
ax.plot(iters, region_C_mass, color=ACCENT[4], lw=2, label='Region C (dangling, 10 nodes)')
ax.plot(iters, cluster_mass,  color=ACCENT[3], lw=2, label='Cluster (semi-isolated, 20 nodes)')
ax.axhline(0, color='#8b949e', ls=':', lw=0.8)
ax.set_xlabel('Iteration'); ax.set_ylabel('Total rank mass')
ax.set_title('Region Rank Mass — Cycle Trap Wins')
ax.legend(fontsize=8); ax.grid(True)

# ── Top-right: individual cycle nodes vs normal nodes ─────────────────────────
ax = axes[0, 1]
for n in [55, 60, 65]:
    h = [history[t][n] for t in range(len(history))]
    ax.plot(iters, h, color=ACCENT[2], lw=1.5, alpha=0.8, label=f'Cycle node {n}' if n==55 else '_')
for n in [5, 15, 25]:
    h = [history[t][n] for t in range(len(history))]
    ax.plot(iters, h, color=ACCENT[0], lw=1.5, alpha=0.8, ls='--', label=f'Normal node {n}' if n==5 else '_')
ax.set_xlabel('Iteration'); ax.set_ylabel('rank(v)')
ax.set_title('Cycle Nodes vs Normal Nodes — Individual Rank')
ax.legend(fontsize=9); ax.grid(True)

# ── Bottom-left: ratio of cycle mass to total surviving mass ──────────────────
ax = axes[1, 0]
surviving = np.array(total_mass)
cycle_arr  = np.array(region_B_mass)
ratio = np.where(surviving > 1e-12, cycle_arr / surviving, 0)
ax.plot(iters, ratio, color=ACCENT[2], lw=2.5)
ax.axhline(20/100, color='#8b949e', ls='--', lw=1,
           label='Fair share (20/100 = 0.20)')
ax.fill_between(iters, 20/100, ratio, where=ratio > 20/100,
                alpha=0.2, color=ACCENT[2], label='Over-representation')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cycle mass / Total surviving mass')
ax.set_title('Rank Inflation in Cycle Trap')
ax.legend(fontsize=9); ax.grid(True)

# ── Bottom-right: heatmap of final rank distribution ─────────────────────────
ax = axes[1, 1]
final_rank = history[-1]
rank_grid = final_rank.reshape(10, 10)
im = ax.imshow(rank_grid, cmap='magma', aspect='auto')
plt.colorbar(im, ax=ax, label='rank(v)')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(range(0, 100, 10), fontsize=7)
ax.set_yticklabels(range(0, 100, 10), fontsize=7)
ax.set_title('Final Rank Distribution (node i = row·10 + col)')

plt.suptitle('Pathology 2: Cycles Pump Probability — Wrong Rank Results',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6 · Pathology 3 — Non-Convergence & Oscillation

Because the graph is **reducible** (not strongly connected), the stationary distribution depends on the starting vector. We demonstrate this by running from two different initial distributions and showing they converge to *different* limiting vectors.

In [ ]:
def power_iter_init(H, r0, max_iter=300):
    """Power iteration with a custom initial vector."""
    r = r0.copy()
    history = [r.copy()]
    for _ in range(max_iter):
        r_new = r @ H
        history.append(r_new.copy())
        if np.linalg.norm(r_new - r, 1) < 1e-10:
            break
        r = r_new
    return np.array(history)

# Init 1: uniform
r_uniform = np.ones(N) / N

# Init 2: all mass on region A
r_regionA = np.zeros(N)
r_regionA[:50] = 1.0 / 50

# Init 3: all mass on cycle nodes
r_cycle = np.zeros(N)
r_cycle[50:70] = 1.0 / 20

hist_u = power_iter_init(H, r_uniform)
hist_A = power_iter_init(H, r_regionA)
hist_C = power_iter_init(H, r_cycle)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = ['Uniform init', 'Region-A init', 'Cycle init']
hists  = [hist_u, hist_A, hist_C]
colors = [ACCENT[0], ACCENT[1], ACCENT[2]]

for ax, h, lbl, col in zip(axes, hists, labels, colors):
    final = h[-1]
    x = np.arange(N)
    node_colors = []
    for n in range(N):
        if n < 50:   node_colors.append(ACCENT[0])
        elif n < 70: node_colors.append(ACCENT[2])
        elif n < 80: node_colors.append(ACCENT[4])
        else:        node_colors.append(ACCENT[3])
    ax.bar(x, final, color=node_colors, alpha=0.85, width=1.0)
    ax.set_xlabel('Node')
    ax.set_ylabel('Final rank')
    ax.set_title(f'{lbl}\n({len(h)-1} iters, mass={final.sum():.4f})')
    ax.grid(True, axis='y')

plt.suptitle('Pathology 3: Different Starts → Different Limits  (Reducible Graph)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# L1 distance between final vectors
print(f'L1 distance (Uniform vs Region-A init): {np.linalg.norm(hist_u[-1] - hist_A[-1], 1):.6f}')
print(f'L1 distance (Uniform vs Cycle init)   : {np.linalg.norm(hist_u[-1] - hist_C[-1], 1):.6f}')
print('\nNon-zero means the "stationary" distribution is NOT unique — broken model!')

## 7 · Focused Node Watch — Four Surgically Selected Nodes

In [ ]:
descriptions = {
    5 : ('Normal node', ACCENT[0]),
    55: ('Cycle trap node', ACCENT[2]),
    73: ('Dangling node', ACCENT[4]),
    85: ('Semi-isolated cluster', ACCENT[3]),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
iters = np.arange(len(history))

for ax, (node, (desc, col)) in zip(axes, descriptions.items()):
    rank_over_time = [history[t][node] for t in range(len(history))]
    fair_share = 1.0 / N

    ax.plot(iters, rank_over_time, color=col, lw=2.5, label=f'rank({node})')
    ax.axhline(fair_share, color='#8b949e', ls='--', lw=1.2, label='Fair share (1/N)')
    ax.fill_between(iters, fair_share, rank_over_time,
                    where=np.array(rank_over_time) > fair_share,
                    alpha=0.15, color=col, label='Above fair share')
    ax.fill_between(iters, fair_share, rank_over_time,
                    where=np.array(rank_over_time) < fair_share,
                    alpha=0.15, color=ACCENT[2], label='Below fair share')

    # Annotate out-degree
    out_d = G.out_degree(node)
    in_d  = G.in_degree(node)
    ax.set_title(f'Node {node} — {desc}\n(in-deg={in_d}, out-deg={out_d})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('rank(v)')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.suptitle('Focused Node Watch — How Each Region Behaves Differently',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8 · Convergence Time vs Graph Size

We sweep graph sizes from 10 to 1000 nodes and measure how many iterations power iteration needs to converge (or stall) under the no-teleportation model.

In [ ]:
def build_similar_graph(n, seed=0):
    """Build a graph with the same structure proportions as our main graph."""
    rng2 = np.random.default_rng(seed)
    G2 = nx.DiGraph()
    G2.add_nodes_from(range(n))
    normal_end  = int(n * 0.50)
    cycle_end   = int(n * 0.70)
    dangle_end  = int(n * 0.80)
    # Region A
    for u in range(normal_end):
        out = rng2.integers(1, min(6, normal_end))
        targets = rng2.choice([v for v in range(normal_end) if v != u],
                              size=min(out, normal_end-1), replace=False)
        for v in targets: G2.add_edge(u, v)
    # Region B: cycle
    cyc = list(range(normal_end, cycle_end))
    for i, u in enumerate(cyc):
        G2.add_edge(u, cyc[(i+1) % len(cyc)])
    # Region C: dangling (no out-edges added)
    for d in range(cycle_end, dangle_end):
        if normal_end > 0:
            G2.add_edge(rng2.integers(0, normal_end), d)
    # Cluster
    for u in range(dangle_end, n):
        cands = [v for v in range(dangle_end, n) if v != u]
        if cands:
            G2.add_edge(u, rng2.choice(cands))
    # Connective tissue
    if normal_end > 0 and len(cyc) > 0:
        for _ in range(max(1, n//20)):
            G2.add_edge(rng2.integers(0, normal_end), rng2.integers(normal_end, max(normal_end+1, cycle_end)))
    return G2


def measure_convergence(G2, tol=1e-8, max_iter=500):
    """Return (iters_to_converge_or_max, final_mass, wall_time_ms)."""
    n = G2.number_of_nodes()
    H2 = build_transition_matrix(G2)
    r = np.ones(n) / n
    t0 = time.perf_counter()
    for i in range(max_iter):
        r_new = r @ H2
        if np.linalg.norm(r_new - r, 1) < tol:
            elapsed = (time.perf_counter() - t0) * 1000
            return i + 1, r_new.sum(), elapsed
        r = r_new
    elapsed = (time.perf_counter() - t0) * 1000
    return max_iter, r.sum(), elapsed   # did not converge → return max


sizes  = [10, 20, 50, 100, 200, 300, 500, 750, 1000]
n_runs = 5   # average over multiple seeds

results = {}
for sz in sizes:
    iters_list, mass_list, time_list = [], [], []
    for seed in range(n_runs):
        G2 = build_similar_graph(sz, seed=seed)
        it, mass, ms = measure_convergence(G2)
        iters_list.append(it); mass_list.append(mass); time_list.append(ms)
    results[sz] = {
        'iters': np.mean(iters_list), 'iters_std': np.std(iters_list),
        'mass' : np.mean(mass_list),
        'time' : np.mean(time_list),  'time_std':  np.std(time_list),
    }
    print(f'  N={sz:5d} → avg iters={results[sz]["iters"]:6.1f}  '
          f'final mass={results[sz]["mass"]:.4f}  time={results[sz]["time"]:6.2f}ms')

In [ ]:
sz_arr    = np.array(sizes)
iter_arr  = np.array([results[s]['iters']     for s in sizes])
iter_std  = np.array([results[s]['iters_std'] for s in sizes])
mass_arr  = np.array([results[s]['mass']      for s in sizes])
time_arr  = np.array([results[s]['time']      for s in sizes])
time_std  = np.array([results[s]['time_std']  for s in sizes])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── 1: Iterations to converge ────────────────────────────────────────────────
ax = axes[0]
ax.errorbar(sz_arr, iter_arr, yerr=iter_std, color=ACCENT[0],
            lw=2, marker='o', markersize=6, capsize=4, label='Mean iters')
ax.set_xlabel('Number of nodes (N)')
ax.set_ylabel('Iterations to converge')
ax.set_title('Convergence Iterations vs Graph Size')
ax.grid(True); ax.legend()

# ── 2: Final probability mass remaining ──────────────────────────────────────
ax = axes[1]
ax.plot(sz_arr, mass_arr, color=ACCENT[2], lw=2, marker='s', markersize=6,
        label='Final mass')
ax.axhline(1.0, color='#8b949e', ls='--', lw=1, label='No leak (ideal)')
ax.fill_between(sz_arr, mass_arr, 1.0, alpha=0.15, color=ACCENT[2], label='Leaked')
ax.set_xlabel('Number of nodes (N)')
ax.set_ylabel('Σ rank(v) at convergence')
ax.set_title('Probability Leakage vs Graph Size')
ax.set_ylim(0, 1.05)
ax.grid(True); ax.legend()

# ── 3: Wall-clock time ────────────────────────────────────────────────────────
ax = axes[2]
ax.errorbar(sz_arr, time_arr, yerr=time_std, color=ACCENT[3],
            lw=2, marker='^', markersize=6, capsize=4, label='Wall time (ms)')
ax.set_xlabel('Number of nodes (N)')
ax.set_ylabel('Time (ms)')
ax.set_title('Computation Time vs Graph Size')
ax.grid(True); ax.legend()

plt.suptitle('Convergence Scaling — Without Teleportation',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9 · Summary & Why Teleportation Fixes This

| Pathology | Root cause | Effect on rank |
|---|---|---|
| **Probability leak** | Dangling nodes absorb flow, emit nothing | Total mass < 1; all ranks → 0 |
| **Rank inflation in cycles** | Sink SCC keeps all incoming mass | Cycle nodes get disproportionate rank |
| **Non-unique limit** | Reducible graph (multiple recurrent classes) | Final rank depends on initial vector |
| **Slow / failed convergence** | Eigenvalue 1 is not the dominant, isolated eigenvalue | Hundreds of iterations or stall |

**The fix — teleportation (damping factor d ≈ 0.85):**

$$
G = d \cdot H_{\text{corrected}} + \frac{(1-d)}{N} \mathbf{1}\mathbf{1}^T
$$

1. The uniform teleportation term replaces all zero-rows (dangling fix), **preventing probability leak**.
2. The $(1-d)$ factor injects fresh rank from *every* node into *every* other node, **breaking rank traps**.
3. The resulting Google matrix $G$ is **primitive** (irreducible + aperiodic) → unique stationary distribution by Perron–Frobenius.
4. The spectral gap becomes $1 - d = 0.15$ → convergence in $O(\log(\varepsilon) / \log(d))$ iterations ≈ **50–100 iterations regardless of N**.

In [ ]:
# Quick demonstration: with teleportation, all pathologies vanish
d = 0.85

def build_google_matrix(G, d=0.85):
    n = G.number_of_nodes()
    H_raw = build_transition_matrix(G)
    # Fix dangling rows → uniform
    dangling_mask = H_raw.sum(axis=1) == 0
    H_raw[dangling_mask] = 1.0 / n
    return d * H_raw + (1 - d) / n * np.ones((n, n))

G_mat = build_google_matrix(G, d)
hist_tp, mass_tp, _ = power_iteration(G_mat, max_iter=300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(np.arange(len(mass_tp)), mass_tp, color=ACCENT[1], lw=2.5, label='With teleportation')
ax.plot(np.arange(len(total_mass)), total_mass, color=ACCENT[2], lw=2.5, ls='--', label='Without teleportation')
ax.axhline(1.0, color='#8b949e', ls=':', lw=1)
ax.set_xlabel('Iteration'); ax.set_ylabel('Total mass')
ax.set_title('Total Probability Mass: With vs Without Teleportation')
ax.legend(); ax.grid(True)

ax = axes[1]
ax.bar(np.arange(N), hist_tp[-1],  color=ACCENT[1], alpha=0.7, width=1, label='With teleportation')
ax.bar(np.arange(N), history[-1], color=ACCENT[2], alpha=0.5, width=1, label='Without teleportation')
ax.set_xlabel('Node'); ax.set_ylabel('rank(v)')
ax.set_title('Final Rank Distribution Comparison')
ax.legend(fontsize=9); ax.grid(True, axis='y')

print(f'Without teleportation: converged in {len(history)-1} iters, final mass = {total_mass[-1]:.6f}')
print(f'With    teleportation: converged in {len(hist_tp)-1} iters, final mass = {mass_tp[-1]:.6f}')

plt.suptitle('Teleportation Fixes All Three Pathologies', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()